# 01 – Đọc dữ liệu IMDb từ CSV

**Đầu vào:** Một hoặc nhiều CSV đặt trong `data/raw/` (dataset [IMDb Dataset 2024 updated](https://www.kaggle.com/datasets/parthdande/imdb-dataset-2024-updated)).

**Đầu ra:** `data/processed/imdb_raw_merged.csv` và `data/processed/source_inventory.csv`.

Chạy tất cả cell theo thứ tự, từ thư mục dự án hoặc thư mục `notebooks/`.

In [3]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 50)
print('Thư mục dự án:', ROOT)

Thư mục dự án: e:\Nhập Môn AI\NhapMonPhanTichDuLieuVaAI-FinalProject


## 1.1. Tìm và đọc các CSV

Đọc toàn bộ CSV trong `data/raw`, chỉ chuẩn hóa tên cột giữa các file. **Không sửa file gốc.**

In [4]:
# Danh sách tên cột có thể khác nhau giữa các file/phát hành.
ALIASES = {
    'title': ['Title', 'Movie Title', 'Movie Name', 'primaryTitle', 'Name'],
    'imdb_rating': ['IMDb Rating', 'IMDB Rating', 'IMDb_Rating', 'Rating', 'IMDB Score', 'averageRating'],
    'year': ['Year', 'Release Year', 'Release_Year', 'startYear', 'Title Year'],
    'duration_minutes': ['Duration (minutes)', 'Duration', 'Duration Minutes', 'Runtime', 'Runtime (Minutes)', 'runtimeMinutes'],
    'metascore': ['MetaScore', 'Metascore', 'Meta Score'],
    'certificates': ['Certificates', 'Certificate', 'Certification', 'Age Rating'],
    'director': ['Director', 'Directors'],
    'star_cast': ['Star Cast', 'StarCast', 'Cast', 'Actors', 'Stars'],
    'poster_url': ['Poster-src', 'Poster URL', 'Poster', 'Poster-src URL'],
    'genres': ['Genres'],
    'genre_1': ['Genre', 'Genre 1', 'First Genre', 'First_Genre'],
    'genre_2': ['Second Genre', 'Second_Genre', 'Genre 2'],
    'genre_3': ['Third Genre', 'Third_Genre', 'Genre 3'],
    'imdb_id': ['imdb_id', 'tconst', 'IMDb ID', 'IMDb_ID'],
}

def simple_key(value):
    value = unicodedata.normalize('NFKD', str(value))
    return re.sub(r'[^a-z0-9]', '', value.lower())

def standardize_headers(frame):
    frame = frame.copy()
    for canonical, variants in ALIASES.items():
        acceptable = {simple_key(v) for v in [canonical] + variants}
        matches = [c for c in frame.columns if simple_key(c) in acceptable]
        if not matches:
            continue
        # Nhiều tên tương đương trong cùng CSV: ưu tiên giá trị có sẵn.
        combined = frame[matches].bfill(axis=1).iloc[:, 0]
        frame = frame.drop(columns=matches)
        frame[canonical] = combined
    return frame

files = sorted(RAW_DIR.glob('*.csv'))
if not files:
    raise FileNotFoundError(
        f'Chưa có file CSV trong {RAW_DIR}. Tải dữ liệu Kaggle rồi chép CSV vào data/raw/.'
    )

parts = []
inventory = []
for file in files:
    try:
        item = pd.read_csv(file, low_memory=False, encoding='utf-8-sig',
                           na_values=['\\N', 'N/A', 'NULL', 'null', '--', ''])
    except UnicodeDecodeError:
        item = pd.read_csv(file, low_memory=False, encoding='cp1252',
                           na_values=['\\N', 'N/A', 'NULL', 'null', '--', ''])
    inventory.append({'file': file.name, 'rows': len(item), 'columns': len(item.columns)})
    if len(item):
        item = standardize_headers(item)
        item['source_file'] = file.name
        parts.append(item)

if not parts:
    raise ValueError('Các CSV không có dòng dữ liệu hợp lệ.')

df = pd.concat(parts, ignore_index=True, sort=False)
source_inventory = pd.DataFrame(inventory)
display(source_inventory)
print(f'Tổng sau gộp: {df.shape[0]:,} dòng × {df.shape[1]} cột')
display(df.head(5))

,file,rows,columns
0,IMDb_Dataset.csv,400,9
1,IMDb_Dataset_2.csv,3173,10
2,IMDb_Dataset_3.csv,1000,12


Tổng sau gộp: 4,573 dòng × 13 cột


,title,imdb_rating,year,duration_minutes,metascore,certificates,director,star_cast,genre_1,source_file,poster_url,genre_2,genre_3
0,Gladiator,8.5,2000,155.0,67.0,R,Ridley Scott,David FranzoniJohn LoganWilliam Nicholson,Action,IMDb_Dataset.csv,NaN,NaN,NaN
1,Mission: Impossible - Dead Reckoning Part One,7.7,2023,163.0,81.0,PG-13,Christopher McQuarrie,Tom CruiseHayley AtwellVing Rhames,Action,IMDb_Dataset.csv,NaN,NaN,NaN
2,Rebel Moon - Part Two: The Scargiver,5.2,2024,122.0,35.0,PG-13,Zack Snyder,Zack SnyderKurt JohnstadShay Hatten,Action,IMDb_Dataset.csv,NaN,NaN,NaN
3,Inglourious Basterds,8.4,2009,153.0,69.0,R,Quentin Tarantino,Brad PittDiane KrugerEli Roth,Adventure,IMDb_Dataset.csv,NaN,NaN,NaN
4,Borderlands,7.1,2024,116.3,66.9,R,Eli Roth,Eli RothJoe Crombie,Action,IMDb_Dataset.csv,NaN,NaN,NaN


## 1.2. Kiểm tra cấu trúc và chất lượng ban đầu

In [5]:
print('Số dòng:', len(df))
print('Số cột:', len(df.columns))
print('Tên các cột:', df.columns.tolist())
print('\nKIỂU DỮ LIỆU:')
df.info()
print('\nTHỐNG KÊ MÔ TẢ:')
display(df.describe(include='all').T)

Số dòng: 4573
Số cột: 13
Tên các cột: ['title', 'imdb_rating', 'year', 'duration_minutes', 'metascore', 'certificates', 'director', 'star_cast', 'genre_1', 'source_file', 'poster_url', 'genre_2', 'genre_3']

KIỂU DỮ LIỆU:
<class 'pandas.DataFrame'>
RangeIndex: 4573 entries, 0 to 4572
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             4573 non-null   str    
 1   imdb_rating       4573 non-null   float64
 2   year              4573 non-null   int64  
 3   duration_minutes  4573 non-null   float64
 4   metascore         4573 non-null   float64
 5   certificates      4573 non-null   str    
 6   director          4573 non-null   str    
 7   star_cast         4573 non-null   str    
 8   genre_1           4573 non-null   str    
 9   source_file       4573 non-null   str    
 10  poster_url        4173 non-null   str    
 11  genre_2           1000 non-null   str    
 12  genre_3           1

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
title,4573,3656,Kung Fu Panda 4,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
imdb_rating,4573.0,NaN,NaN,NaN,6.83466,0.878024,3.9,6.3,6.9,7.4,9.5
year,4573.0,NaN,NaN,NaN,2005.460529,17.263552,1917.0,1997.0,2010.0,2018.0,2025.0
duration_minutes,4573.0,NaN,NaN,NaN,112.749421,22.092234,46.0,99.0,115.0,119.0,317.0
metascore,4573.0,NaN,NaN,NaN,64.122589,13.169511,11.0,57.1,66.0,69.0,100.0
certificates,4573,18,R,1985,NaN,NaN,NaN,NaN,NaN,NaN,NaN
director,4573,2603,Ridley Scott,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN
star_cast,4573,3427,Daniel RadcliffeEmma WatsonRupert Grint,149,NaN,NaN,NaN,NaN,NaN,NaN,NaN
genre_1,4573,18,Biography,894,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source_file,4573,3,IMDb_Dataset_2.csv,3173,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
missing = pd.DataFrame({
    'so_thieu': df.isna().sum(),
    'ty_le_thieu_%': (df.isna().mean() * 100).round(2),
}).sort_values('ty_le_thieu_%', ascending=False)
print('Bảng dữ liệu thiếu:')
display(missing)

same_row = df.drop(columns=['source_file'], errors='ignore').duplicated().sum()
print('Số dòng giống nhau hoàn toàn (không tính source_file):', int(same_row))
if 'title' in df.columns:
    print('Số tiêu đề bị thiếu:', int(df['title'].isna().sum()))

Bảng dữ liệu thiếu:


,so_thieu,ty_le_thieu_%
genre_2,3573,78.13
genre_3,3573,78.13
poster_url,400,8.75
imdb_rating,0,0.00
title,0,0.00
metascore,0,0.00
duration_minutes,0,0.00
year,0,0.00
certificates,0,0.00
genre_1,0,0.00


Số dòng giống nhau hoàn toàn (không tính source_file): 556
Số tiêu đề bị thiếu: 0


## 1.3. Xuất dữ liệu để notebook 02 xử lý

In [7]:
raw_path = OUT_DIR / 'imdb_raw_merged.csv'
df.to_csv(raw_path, index=False, encoding='utf-8-sig')
source_inventory.to_csv(OUT_DIR / 'source_inventory.csv', index=False, encoding='utf-8-sig')
print('Đã tạo:', raw_path)

Đã tạo: e:\Nhập Môn AI\NhapMonPhanTichDuLieuVaAI-FinalProject\data\processed\imdb_raw_merged.csv
